# 🚀 BoneRAG — Batch Multi-Model GPU Indexing for FracAtlas Dataset

Notebook này tự động hóa việc chạy **Batch GPU Indexing 1 LẦN DUY NHẤT** cho **TOÀN BỘ 5 Foundation Models** nổi tiếng nhất trên tập dữ liệu **FracAtlas (4,082 ảnh X-quang)** bằng Google Colab T4 GPU miễn phí.

### 🏆 Danh sách 5 Foundation Models được mã hóa song song:
1. 🔬 **BiomedCLIP (Microsoft)** — `fracatlas_biomedclip.faiss` (PubMedBERT + ViT-B/16 y khoa chuyên sâu)
2. 👁️ **OpenAI CLIP ViT-B/32** — `fracatlas_clip_vitb32.faiss` (General CLIP tiêu chuẩn)
3. 🚀 **OpenAI CLIP ViT-L/14** — `fracatlas_clip_vitl14.faiss` (Large Vision CLIP)
4. 🌐 **LAION-2B CLIP ViT-H/14** — `fracatlas_clip_vith14.faiss` (Huge 1B+ Vision Transformer)
5. ⚡ **BioViL-T / PubMed ViT** — `fracatlas_biovil.faiss` (Chest & Limb X-ray CLIP)

### 📌 Hướng dẫn:
1. **Bật GPU**: Menu `Runtime` -> `Change runtime type` -> Chọn `T4 GPU`.
2. Bấm **Runtime -> Run all (Ctrl + F9)**.
3. Chờ ~3 phút, hệ thống tự động xuất đủ **10 tệp FAISS Index & Metadata** của cả 5 mô hình!

In [ ]:
# [Step 1] Cài đặt thư viện tối ưu hóa GPU trên Colab
!pip install -q torch open_clip_torch faiss-cpu pillow tqdm huggingface_hub

In [ ]:
# [Step 2] Tải toàn bộ 4,082 ảnh X-quang Dataset FracAtlas về Colab
import os
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/BoneRAG_Data")
INDEX_STORE_DIR = DRIVE_DIR / "indexes"
MODEL_CACHE_DIR = DRIVE_DIR / "model_cache"
USE_DRIVE = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    for folder in (DRIVE_DIR, INDEX_STORE_DIR, MODEL_CACHE_DIR):
        folder.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(MODEL_CACHE_DIR / "huggingface")
    os.environ["HUGGINGFACE_HUB_CACHE"] = str(MODEL_CACHE_DIR / "huggingface")
    os.environ["TORCH_HOME"] = str(MODEL_CACHE_DIR / "torch")
    USE_DRIVE = True
except Exception:
    print("Không dùng được Google Drive -> index chỉ tồn tại trong runtime hiện tại.")

print("📥 Đang tải toàn bộ bộ dữ liệu FracAtlas (4,082 ảnh X-quang)... ")
drive_zip = DRIVE_DIR / "fracatlas_full.zip"
local_zip = Path("fracatlas_full.zip")
if USE_DRIVE and drive_zip.exists():
    !cp "{drive_zip}" "{local_zip}"
else:
    target_zip = drive_zip if USE_DRIVE else local_zip
    !wget -q --show-progress -O "{target_zip}" "https://huggingface.co/datasets/runananya/fracatlas/resolve/main/archive%20%289%29.zip" || curl -L -o "{target_zip}" "https://figshare.com/ndownloader/articles/22276042/versions/1"

print("📦 Đang giải nén dữ liệu...")
!unzip -q -o "{local_zip}" -d ./fracatlas_repo || true

image_files = list(Path("./fracatlas_repo").rglob("*.jpg")) + list(Path("./fracatlas_repo").rglob("*.jpeg")) + list(Path("./fracatlas_repo").rglob("*.png"))
print(f"🎉 ĐÃ TÌM THẤY {len(image_files)} TỆP ẢNH X-QUANG THẬT TRONG FRACATLAS!")

In [ ]:
# [Step 3] Định nghĩa Danh sách 5 Foundation Models
import torch
import open_clip
import faiss
import json
import numpy as np
from PIL import Image
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ GPU Device: {device} ({torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'})")

# Cấu hình 5 Foundation Models
FOUNDATION_MODELS = [
    {
        "name": "BiomedCLIP (Microsoft Medical SOTA)",
        "prefix": "fracatlas_biomedclip",
        "hub": "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
    },
    {
        "name": "OpenAI CLIP ViT-B/32 (Base)",
        "prefix": "fracatlas_clip_vitb32",
        "model": "ViT-B-32",
        "pretrained": "openai"
    },
    {
        "name": "OpenAI CLIP ViT-L/14 (Large)",
        "prefix": "fracatlas_clip_vitl14",
        "model": "ViT-L-14",
        "pretrained": "openai"
    },
    {
        "name": "LAION-2B CLIP ViT-H/14 (Huge)",
        "prefix": "fracatlas_clip_vith14",
        "model": "ViT-H-14",
        "pretrained": "laion2b_s32b_b79k"
    },
    {
        "name": "BioViL / PubMed ViT-B/16",
        "prefix": "fracatlas_biovil",
        "model": "ViT-B-16",
        "pretrained": "laion2b_s34b_b88k"
    }
]
print(f"📋 Đã đăng ký {len(FOUNDATION_MODELS)} Foundation Models!")

In [ ]:
# [Step 4] Chạy Batch GPU Encoding tự động cho CẢ 5 MODELS cùng lúc!
print(f"🚀 BẮT ĐẦU MÃ HÓA TỰ ĐỘNG CẢ {len(FOUNDATION_MODELS)} MODELS CHO {len(image_files)} ẢNH X-QUANG...\n")

BATCH_SIZE = 64

for model_idx, m_cfg in enumerate(FOUNDATION_MODELS, 1):
    model_name = m_cfg["name"]
    prefix = m_cfg["prefix"]
    print(f"\n---------------------------------------------------------------")
    print(f"[{model_idx}/{len(FOUNDATION_MODELS)}] ⚙️ Đang xử lý Mô hình: {model_name}")
    print(f"---------------------------------------------------------------")
    
    try:
        if "hub" in m_cfg:
            model, _, preprocess = open_clip.create_model_and_transforms(m_cfg["hub"])
        else:
            model, _, preprocess = open_clip.create_model_and_transforms(m_cfg["model"], pretrained=m_cfg["pretrained"])
        
        model.to(device).eval()
        
        vectors = []
        metadata = []
        
        # Batch processing to maximize GPU throughput
        for i in range(0, len(image_files), BATCH_SIZE):
            batch_paths = image_files[i:i + BATCH_SIZE]
            tensors = []
            valid_paths = []
            
            for img_path in batch_paths:
                try:
                    img = Image.open(img_path).convert("RGB")
                    tensors.append(preprocess(img))
                    valid_paths.append(img_path)
                except Exception:
                    continue
            
            if not tensors:
                continue
            
            batch_tensor = torch.stack(tensors).to(device)
            with torch.no_grad():
                feats = model.encode_image(batch_tensor)
                feats /= feats.norm(dim=-1, keepdim=True)
                feats_np = feats.cpu().numpy().astype(np.float32)
            
            for p, vec in zip(valid_paths, feats_np):
                parent_label = p.parent.name.lower().replace("-", "_")
                is_frac = parent_label == "fractured"
                vectors.append(vec)
                metadata.append({
                    "image_id": f"fracatlas-{'fractured' if is_frac else 'normal'}-{p.stem.lower()}",
                    "title": f"FracAtlas X-ray {p.name}",
                    "body_part": "forearm/wrist",
                    "diagnosis": "fracture" if is_frac else "normal",
                    "fracture_type": "fractured" if is_frac else "none",
                    "region": "forearm and wrist",
                    "evidence_note": f"FracAtlas real X-ray dataset case {p.name}.",
                    "text": f"fracatlas {'fractured' if is_frac else 'normal'} xray wrist forearm bone case {p.stem.lower()}",
                    "image_path": str(p)
                })
        
        # Save FAISS Index & Metadata for this model
        vec_matrix = np.array(vectors, dtype=np.float32)
        dim = vec_matrix.shape[1]
        
        index = faiss.IndexFlatIP(dim)
        faiss.normalize_L2(vec_matrix)
        index.add(vec_matrix)
        
        faiss_file = f"{prefix}.faiss"
        meta_file = f"{prefix}_metadata.json"
        
        faiss.write_index(index, faiss_file)
        with open(meta_file, "w", encoding="utf-8") as fh:
            json.dump(metadata, fh, ensure_ascii=False, indent=2)
            
        print(f"✅ [Thành công] {model_name} -> {faiss_file} ({dim}-dim, {len(vectors)} vectors)")
        
        # Clean GPU memory before loading next model
        del model
        torch.cuda.empty_cache()
        
    except Exception as exc:
        print(f"❌ Lỗi khi xử lý model {model_name}: {exc}")

In [ ]:
# [Step 5] Tổng kết danh sách Tệp xuất ra
print("\n=======================================================")
print("🎉 HOÀN THÀNH TẤT CẢ 5 FOUNDATION MODELS!")
print("📁 Các tệp FAISS Index & Metadata xuất tại Colab:")
for m in FOUNDATION_MODELS:
    p = m["prefix"]
    f_faiss = f"{p}.faiss"
    f_meta = f"{p}_metadata.json"
    e_faiss = "✅" if os.path.exists(f_faiss) else "❌"
    print(f"  {e_faiss} {f_faiss} & {f_meta}")
if USE_DRIVE:
    from shutil import copy2
    INDEX_STORE_DIR.mkdir(parents=True, exist_ok=True)
    for m in FOUNDATION_MODELS:
        for suffix in (".faiss", "_metadata.json"):
            artifact = Path(m["prefix"] + suffix)
            if artifact.exists():
                copy2(artifact, INDEX_STORE_DIR / artifact.name)
    print(f"Đã lưu FAISS/metadata vào {INDEX_STORE_DIR}")
print("=======================================================")